# Long-Tailed Object Detection: YOLOv11 adds-on attention with Re-Balancing

This notebook anchors the baseline training pipeline for drone-view object detection using YOLOv11 **without any pretrained weights**.
Fill in each `# TODO` placeholder before executing the training cell below.


In [1]:
from pathlib import Path
from datetime import datetime
from typing import List, Tuple, Optional
import random
import shutil

from PIL import Image
import yaml

PROJECT_DIR = Path.cwd()
DATA_ROOT = (PROJECT_DIR / '../../../dataset/taica-cvpdl-2025-hw-2/CVPDL_hw2/CVPDL_hw2').resolve()
TRAIN_DIR = DATA_ROOT / 'train'
TEST_DIR = DATA_ROOT / 'test'

if not TRAIN_DIR.exists():
    raise FileNotFoundError(f'Dataset train folder not found at {TRAIN_DIR}. Update DATA_ROOT if your layout differs.')

if not TEST_DIR.exists():
    print(f'Warning: test folder not found at {TEST_DIR}. Test images will not be copied.')
    TEST_DIR = None

EXPERIMENT_NAME = 'yolov10_attention'
RUN_ID = datetime.now().strftime('%Y%m%d-%H%M%S')
EXPERIMENT_DIR = (PROJECT_DIR / 'artifacts' / EXPERIMENT_NAME / RUN_ID).resolve()
YOLO_DATA_DIR = EXPERIMENT_DIR / 'dataset'
RUN_ROOT_DIR = EXPERIMENT_DIR / 'run'
TRAIN_RUN_NAME = 'train'
VAL_RUN_NAME = 'val'
INFER_RUN_NAME = 'infer'

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
RUN_ROOT_DIR.mkdir(parents=True, exist_ok=True)

VAL_RATIO = 0.2  # fixed 80/20 train/val split
SEED = 11


def load_label_file(label_path: Path) -> List[Tuple[int, float, float, float, float]]:
    boxes: List[Tuple[int, float, float, float, float]] = []
    with label_path.open('r') as fh:
        for raw_line in fh:
            line = raw_line.strip()
            if not line:
                continue
            parts = [p.strip() for p in line.split(',')]
            if len(parts) < 5:
                continue
            cls = int(parts[0])
            x, y, w, h = map(float, parts[1:5])
            boxes.append((cls, x, y, w, h))
    return boxes


def convert_tlwh_to_yolo(x: float, y: float, w: float, h: float, img_w: int, img_h: int) -> Tuple[float, float, float, float]:
    xc = (x + w / 2.0) / img_w
    yc = (y + h / 2.0) / img_h
    ww = w / img_w
    hh = h / img_h
    xc = min(max(xc, 0.0), 1.0)
    yc = min(max(yc, 0.0), 1.0)
    ww = min(max(ww, 0.0), 1.0)
    hh = min(max(hh, 0.0), 1.0)
    return xc, yc, ww, hh


def prepare_dataset(train_dir: Path, output_dir: Path, val_ratio: float, seed: int, test_dir: Optional[Path] = None):
    if output_dir.exists():
        shutil.rmtree(output_dir)
    images_root = output_dir / 'images'
    labels_root = output_dir / 'labels'
    for split in ['train', 'val']:
        (images_root / split).mkdir(parents=True, exist_ok=True)
        (labels_root / split).mkdir(parents=True, exist_ok=True)
    if test_dir is not None:
        (images_root / 'test').mkdir(parents=True, exist_ok=True)
        (labels_root / 'test').mkdir(parents=True, exist_ok=True)

    image_stems = sorted({path.stem for path in train_dir.glob('*.png')})
    missing_labels: List[str] = []
    missing_images: List[str] = []

    for label_path in train_dir.glob('*.txt'):
        if not (train_dir / f'{label_path.stem}.png').exists():
            missing_images.append(label_path.stem)

    records: List[Tuple[str, Path, Path]] = []
    for stem in image_stems:
        image_path = train_dir / f'{stem}.png'
        label_path = train_dir / f'{stem}.txt'
        if not label_path.exists():
            missing_labels.append(stem)
            continue
        records.append((stem, image_path, label_path))

    rng = random.Random(seed)
    rng.shuffle(records)

    if not records:
        raise RuntimeError('No image/label pairs found in the training dataset.')

    if len(records) == 1:
        val_count = 0
    else:
        val_count = max(1, int(len(records) * val_ratio))
        val_count = min(len(records) - 1, val_count)

    val_stems = {stem for stem, *_ in records[:val_count]}

    stats = {
        'train': 0,
        'val': 0,
        'test': 0,
        'boxes': 0,
        'missing_labels': missing_labels,
        'missing_images': missing_images,
        'class_ids': set(),
    }

    for stem, image_path, label_path in records:
        split = 'val' if stem in val_stems else 'train'
        with Image.open(image_path) as img:
            img_w, img_h = img.size

        label_entries = load_label_file(label_path)
        yolo_lines = []
        for cls, x, y, w, h in label_entries:
            xc, yc, ww, hh = convert_tlwh_to_yolo(x, y, w, h, img_w, img_h)
            yolo_lines.append(f'{cls} {xc:.6f} {yc:.6f} {ww:.6f} {hh:.6f}')
            stats['class_ids'].add(cls)

        dst_img = images_root / split / f'{stem}.png'
        shutil.copy2(image_path, dst_img)

        dst_label = labels_root / split / f'{stem}.txt'
        dst_label.write_text('\n'.join(yolo_lines))

        stats[split] += 1
        stats['boxes'] += len(yolo_lines)

    if test_dir is not None:
        test_images = sorted(test_dir.glob('*.png'))
        for image_path in test_images:
            shutil.copy2(image_path, images_root / 'test' / image_path.name)
            stats['test'] += 1

    stats['val_ratio'] = val_ratio
    return stats


stats = prepare_dataset(
    TRAIN_DIR,
    YOLO_DATA_DIR,
    val_ratio=VAL_RATIO,
    seed=SEED,
    test_dir=TEST_DIR,
)

print(f'Dataset prepared at {YOLO_DATA_DIR}')
print(f"train images: {stats['train']} | val images: {stats['val']} | boxes: {stats['boxes']}")
if stats['test']:
    print(f"test images copied: {stats['test']}")
if stats['missing_labels']:
    sample = ', '.join(stats['missing_labels'][:5])
    print(f"Skipped {len(stats['missing_labels'])} images without labels. Examples: {sample}")
if stats['missing_images']:
    sample = ', '.join(stats['missing_images'][:5])
    print(f"Skipped {len(stats['missing_images'])} labels without images. Examples: {sample}")

class_ids = sorted(stats['class_ids'])
if not class_ids:
    raise RuntimeError('No class ids found in annotations. Check label parsing logic.')

ALL_CLASS_NAMES = {
    0: 'car',
    1: 'hov',
    2: 'person',
    3: 'motorcycle',
}
missing_class_ids = sorted(set(class_ids) - set(ALL_CLASS_NAMES))
if missing_class_ids:
    raise ValueError(f'Unknown class ids {missing_class_ids} detected. Update ALL_CLASS_NAMES mapping to include them.')

CLASS_NAME_MAP = {cid: ALL_CLASS_NAMES[cid] for cid in class_ids}
DATA_CONFIG_PATH = EXPERIMENT_DIR / 'longtail_dataset.yaml'

data_yaml = {
    'path': YOLO_DATA_DIR.as_posix(),
    'train': 'images/train',
    'val': 'images/val',
}
if stats['test']:
    data_yaml['test'] = 'images/test'
data_yaml['nc'] = len(CLASS_NAME_MAP)
data_yaml['names'] = {cid: name for cid, name in CLASS_NAME_MAP.items()}

with DATA_CONFIG_PATH.open('w') as fh:
    yaml.safe_dump(data_yaml, fh, sort_keys=False)

print(f'YOLO data config saved to {DATA_CONFIG_PATH}')




Dataset prepared at /home/eclab314/314706007/2025-CVPDL/HW2/hw2_314706007/code_314706007/src/artifacts/yolov10_attention/20251031-195048/dataset
train images: 760 | val images: 190 | boxes: 33331
test images copied: 550
YOLO data config saved to /home/eclab314/314706007/2025-CVPDL/HW2/hw2_314706007/code_314706007/src/artifacts/yolov10_attention/20251031-195048/longtail_dataset.yaml


In [2]:
from collections import Counter
import math


def load_yolo_label_entries(label_path: Path) -> list[tuple[int, float, float, float, float]]:
    entries = []
    if not label_path.exists():
        return entries
    for raw in label_path.read_text().strip().splitlines():
        raw = raw.strip()
        if not raw:
            continue
        parts = raw.split()
        if len(parts) < 5:
            continue
        cls = int(parts[0])
        x, y, w, h = map(float, parts[1:5])
        entries.append((cls, x, y, w, h))
    return entries


train_images_dir = YOLO_DATA_DIR / 'images/train'
train_labels_dir = YOLO_DATA_DIR / 'labels/train'
val_labels_dir = YOLO_DATA_DIR / 'labels/val'

if not train_labels_dir.exists():
    raise FileNotFoundError(f'Train labels directory not found at {train_labels_dir}')

TRAIN_RECORDS: list[dict] = []
CLASS_BOX_COUNTS: Counter[int] = Counter()
CLASS_IMAGE_COUNTS: Counter[int] = Counter()
TOTAL_BOXES = 0

for label_path in sorted(train_labels_dir.glob('*.txt')):
    stem = label_path.stem
    boxes = load_yolo_label_entries(label_path)
    class_hist = Counter(box[0] for box in boxes)
    if not class_hist:
        continue
    TRAIN_RECORDS.append(
        {
            'stem': stem,
            'image_path': train_images_dir / f'{stem}.png',
            'label_path': label_path,
            'class_hist': class_hist,
            'box_count': sum(class_hist.values()),
        }
    )
    CLASS_BOX_COUNTS.update(class_hist)
    CLASS_IMAGE_COUNTS.update(class_hist.keys())
    TOTAL_BOXES += sum(class_hist.values())

NUM_CLASSES = len(CLASS_NAME_MAP)
if NUM_CLASSES == 0:
    raise RuntimeError('CLASS_NAME_MAP is empty; dataset preparation may have failed.')

max_count = max(CLASS_BOX_COUNTS.values())
min_count = min(CLASS_BOX_COUNTS.values())
IMBALANCE_RATIO = max_count / max(1, min_count)

print('Class distribution (box counts):')
for cls_id in sorted(CLASS_NAME_MAP):
    name = CLASS_NAME_MAP[cls_id]
    box_count = CLASS_BOX_COUNTS.get(cls_id, 0)
    image_count = CLASS_IMAGE_COUNTS.get(cls_id, 0)
    print(f'  class {cls_id} ({name:>10}): boxes={box_count:5d}, images_with_class={image_count:4d}')

print(f'Total boxes: {TOTAL_BOXES}')
print(f'Imbalance ratio (max/min): {IMBALANCE_RATIO:.3f}')

BASE_DATASET_DIR = YOLO_DATA_DIR
BASE_DATASET_CLASS_COUNTS = CLASS_BOX_COUNTS.copy()
BASE_IMBALANCE_RATIO = IMBALANCE_RATIO


Class distribution (box counts):
  class 0 (       car): boxes=19023, images_with_class= 746
  class 1 (       hov): boxes= 1118, images_with_class= 462
  class 2 (    person): boxes= 2732, images_with_class= 404
  class 3 (motorcycle): boxes= 4264, images_with_class= 400
Total boxes: 27137
Imbalance ratio (max/min): 17.015


In [3]:
from __future__ import annotations

import random
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import torch
from ultralytics import YOLO
from cbam import C2f_CBAM, CBAMLayer, load_scaled_yaml
from ultralytics.nn import modules as yolo_modules
from ultralytics.nn import tasks as yolo_tasks

# Register CBAM modules so Ultralytics can construct the network
yolo_modules.C2f_CBAM = C2f_CBAM
yolo_modules.CBAMLayer = CBAMLayer
yolo_tasks.C2f_CBAM = C2f_CBAM
yolo_tasks.CBAMLayer = CBAMLayer

try:
    import multiprocessing as mp
    mp.set_start_method('fork', force=True)
except RuntimeError:
    pass



@dataclass
class TrainConfig:
    """Configuration container for baseline YOLOv10 training."""

    # TODO: choose the custom YOLOv10 architecture with attention modules
    model_yaml: str = 'yolov10_CBAM.yaml'

    # Dataset YAML generated in the preparation cell above
    dataset_yaml: Path = DATA_CONFIG_PATH

    # TODO: adjust batch size based on GPU memory
    batch_size: int = 16

    # TODO: adjust epoch count based on convergence needs
    epochs: int = 30

    image_size: int = 640

    # Use CUDA if available; override with a specific device string if needed (e.g. "0", "0,1", "cpu")
    device: str = 'auto'

    workers: int = 8
    project_dir: Path = RUN_ROOT_DIR
    run_name: str = TRAIN_RUN_NAME
    val_name: str = VAL_RUN_NAME
    seed: int = SEED

    # Optimization hyperparameters for a plain baseline run
    optimizer: str = 'SGD'
    learning_rate: float = 0.001
    final_lr_ratio: float = 0.01  # ratio between final and initial LR (Ultralytics uses cosine by default)
    weight_decay: float = 5e-4
    momentum: float = 0.937
    warmup_epochs: float = 3.0

    # Early stopping patience in epochs; adjust if you need longer training
    patience: int = 50

    # Optional: resume from an earlier run (leave as None for a fresh start)
    resume_checkpoint: Optional[Path] = None


def set_deterministic(seed: int) -> None:
    """Set random seeds for reproducible training."""

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        # warn_only missing on older PyTorch; allow nondeterministic ops in that case
        torch.use_deterministic_algorithms(False)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def validate_paths(config: TrainConfig) -> None:
    """Ensure critical input files exist before launching training."""

    if '#TODO' in config.model_yaml:
        raise ValueError('Update TrainConfig.model_yaml with the YOLOv11 architecture name or YAML path you intend to use.')

    dataset_yaml_path = Path(config.dataset_yaml)
    if not dataset_yaml_path.exists():
        raise FileNotFoundError(f'Dataset YAML not found at: {dataset_yaml_path}')


    model_yaml_path = Path(config.model_yaml)
    if model_yaml_path.suffix in {'.yaml', '.yml'}:
        try:
            _resolve_model_yaml_path(model_yaml_path)
        except FileNotFoundError as exc:
            raise FileNotFoundError(f'Model YAML not found for requested path: {model_yaml_path}') from exc
    else:
        if not model_yaml_path.exists():
            candidate = PROJECT_DIR / config.model_yaml
            if not candidate.exists():
                raise FileNotFoundError(f'Model specification not found at: {model_yaml_path}')
    if config.resume_checkpoint is not None:
        resume_path = Path(config.resume_checkpoint)
        if not resume_path.exists():
            raise FileNotFoundError(f'Resume checkpoint not found at: {resume_path}')


In [4]:

import yaml
from pathlib import Path
from typing import Any, Dict, Optional, Tuple


def _extract_scale_hint(stem: str) -> Optional[str]:
    prefix = stem.split('_', 1)[0]
    if prefix and prefix[-1] in {'n', 's', 'm', 'l', 'x'}:
        return prefix[-1]
    return None


def _resolve_model_yaml_path(model_path: Path) -> Tuple[Path, Optional[str]]:
    """Return an existing YAML path and its associated scale hint."""

    def _try_candidates(path: Path) -> Optional[Path]:
        candidates = []
        if path.is_absolute():
            candidates.append(path)
        else:
            candidates.append((PROJECT_DIR / path).resolve())
            candidates.append(path.resolve())
        for candidate in candidates:
            if candidate.is_file():
                return candidate
        return None

    def _find_in_ultralytics(filename: str) -> Optional[Path]:
        try:
            import ultralytics
        except ImportError:
            return None
        cfg_root = Path(ultralytics.__file__).resolve().parent / 'cfg'
        for candidate in cfg_root.rglob(filename):
            if candidate.is_file():
                return candidate
        return None

    resolved = _try_candidates(model_path)
    scale_hint = _extract_scale_hint(model_path.stem)

    if resolved is not None:
        return resolved, scale_hint

    if model_path.suffix in {'.yaml', '.yml'}:
        pkg_resolved = _find_in_ultralytics(model_path.name)
        if pkg_resolved is not None:
            pkg_hint = _extract_scale_hint(pkg_resolved.stem)
            return pkg_resolved, pkg_hint or scale_hint

        stem_parts = model_path.stem.split('_', 1)
        prefix = stem_parts[0]
        suffix_tail = '_' + stem_parts[1] if len(stem_parts) > 1 else ''
        hint = _extract_scale_hint(prefix)
        if hint:
            base_name = prefix[:-1] + suffix_tail + model_path.suffix
            base_path = model_path.with_name(base_name)
            resolved = _try_candidates(base_path)
            if resolved is not None:
                return resolved, hint
            pkg_resolved = _find_in_ultralytics(base_name)
            if pkg_resolved is not None:
                pkg_hint = _extract_scale_hint(pkg_resolved.stem)
                return pkg_resolved, pkg_hint or hint
    raise FileNotFoundError(f'Model YAML not found for: {model_path}')

def build_model(config: TrainConfig, initial_weights: Optional[Path] = None) -> YOLO:
    """Instantiate a YOLOv11 model without loading pretrained weights."""
    if initial_weights is not None:
        model = YOLO(str(initial_weights))
    else:
        model_source = config.model_yaml
        model_path = Path(model_source)
        if model_path.suffix in {".yaml", ".yml"}:
            resolved_path, scale_hint = _resolve_model_yaml_path(model_path)
            scaled_cfg = load_scaled_yaml(resolved_path, scale_hint=scale_hint)
            scaled_yaml_path = config.project_dir / f"{model_path.stem}_resolved.yaml"
            scaled_yaml_path.write_text(yaml.safe_dump(scaled_cfg, sort_keys=False))
            model_source = str(scaled_yaml_path)
        else:
            if not Path(model_source).exists():
                candidate = PROJECT_DIR / model_source
                if candidate.exists():
                    model_source = str(candidate)
        model = YOLO(model_source)
    model.overrides['pretrained'] = False
    return model


def build_train_kwargs(config: TrainConfig, extra_kwargs: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """Generate the argument dictionary passed into `YOLO.train`."""
    kwargs: Dict[str, Any] = {
        'data': str(config.dataset_yaml),
        'epochs': config.epochs,
        'batch': config.batch_size,
        'imgsz': config.image_size,
        'device': config.device,
        'workers': config.workers,
        'project': str(config.project_dir),
        'name': config.run_name,
        'exist_ok': True,
        'optimizer': config.optimizer,
        'lr0': config.learning_rate,
        'lrf': config.final_lr_ratio,
        'weight_decay': config.weight_decay,
        'momentum': config.momentum,
        'warmup_epochs': config.warmup_epochs,
        'patience': config.patience,
        'resume': str(config.resume_checkpoint) if config.resume_checkpoint else False,
        'pretrained': False,
        'save_period': -1,
        'verbose': True,
    }
    if extra_kwargs:
        kwargs.update(extra_kwargs)
    return kwargs


def train_single_stage(
    config: TrainConfig,
    *,
    class_weights: Optional[Dict[int, float]] = None,
    extra_train_kwargs: Optional[Dict[str, Any]] = None,
    initial_weights: Optional[Path] = None,
    alpha_schedule: Optional[Dict[str, float]] = None,
) -> Dict[str, Any]:
    """Train a single stage given a `TrainConfig` and optional overrides."""

    set_deterministic(config.seed)
    validate_paths(config)
    config.project_dir.mkdir(parents=True, exist_ok=True)

    if config.device == 'auto':
        resolved_device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"Auto-selected device: {resolved_device}")
        config.device = resolved_device

    model = build_model(config, initial_weights=initial_weights)
    train_kwargs = build_train_kwargs(config, extra_train_kwargs)
    print(f"Using {train_kwargs.get('workers', 'unknown')} dataloader workers for training")

    if class_weights is not None:
        weight_vector = torch.tensor([class_weights.get(cls, 1.0) for cls in range(len(CLASS_NAME_MAP))], dtype=torch.float32)
        base_vector = torch.ones_like(weight_vector)

        schedule_cfg = alpha_schedule or {}
        start_alpha = float(schedule_cfg.get('start', 1.0))
        end_alpha = float(schedule_cfg.get('end', 1.0))
        start_epoch = int(schedule_cfg.get('start_epoch', 0))
        transition_epochs = int(schedule_cfg.get('transition_epochs', max(config.epochs - start_epoch, 1)))
        decay_power = float(schedule_cfg.get('power', 1.0))
        state = {'alpha': start_alpha}

        def _resolve_alpha(epoch: int) -> float:
            if transition_epochs <= 0:
                return end_alpha
            if epoch < start_epoch:
                return start_alpha
            progress = min(1.0, max(epoch - start_epoch, 0) / max(transition_epochs, 1))
            return start_alpha + (end_alpha - start_alpha) * (progress ** decay_power)

        def _set_weight_vector(trainer) -> None:
            device = getattr(trainer, 'device', None)
            if device is None:
                device = 'cuda' if torch.cuda.is_available() else 'cpu'
            if not isinstance(device, torch.device):
                device = torch.device(device) if isinstance(device, str) else torch.device('cpu')
            alpha = state['alpha']
            blended = (1.0 - alpha) * base_vector + alpha * weight_vector
            weights_device = blended.to(device)
            if hasattr(trainer, 'loss') and hasattr(trainer.loss, 'class_weights'):
                trainer.loss.class_weights = weights_device
            if hasattr(trainer, 'validator') and hasattr(trainer.validator, 'loss_cls') and hasattr(trainer.validator.loss_cls, 'class_weights'):
                trainer.validator.loss_cls.class_weights = weights_device

        def _on_train_start(trainer):
            state['alpha'] = _resolve_alpha(getattr(trainer, 'epoch', 0))
            _set_weight_vector(trainer)

        def _on_train_epoch_start(trainer):
            epoch = getattr(trainer, 'epoch', 0)
            state['alpha'] = _resolve_alpha(epoch)
            if getattr(trainer, 'world_size', 1) in (1, None):
                print(f"[alpha] epoch {epoch}: {state['alpha']:.3f}")
            _set_weight_vector(trainer)

        def _on_val_start(trainer):
            _set_weight_vector(trainer)

        model.add_callback('on_train_start', _on_train_start)
        model.add_callback('on_train_epoch_start', _on_train_epoch_start)
        model.add_callback('on_val_start', _on_val_start)

    train_result = model.train(**train_kwargs)

    run_dir = Path(getattr(train_result, 'save_dir', config.project_dir / config.run_name))
    best_weights_path = run_dir / 'weights' / 'best.pt'
    if best_weights_path.exists():
        print(f'Best checkpoint available at: {best_weights_path}')
    else:
        print(f'Warning: best checkpoint not found at {best_weights_path}')

    return {
        'model': model,
        'train_result': train_result,
        'run_dir': run_dir,
        'best_weights': best_weights_path,
        'config': config,
    }


In [5]:
from pathlib import Path
from typing import Any, Dict, Optional


def run_validation(
    model: YOLO,
    config: TrainConfig,
    *,
    extra_val_kwargs: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """Run validation for a trained YOLO model and capture artifact paths."""

    val_kwargs: Dict[str, Any] = {
        'data': str(config.dataset_yaml),
        'imgsz': config.image_size,
        'batch': config.batch_size,
        'device': config.device,
        'workers': 0,
        'split': 'val',
        'save_json': False,
        'plots': True,
        'verbose': True,
        'project': str(config.project_dir),
        'name': config.val_name,
        'exist_ok': True,
    }
    if extra_val_kwargs:
        val_kwargs.update(extra_val_kwargs)
    val_kwargs['workers'] = 0

    val_results = model.val(**val_kwargs)
    val_dir = Path(val_results.save_dir)
    print(f"Validation metrics saved under: {val_dir}")

    return {
        'val_results': val_results,
        'val_dir': val_dir,
    }


In [6]:
import csv
from pathlib import Path
from typing import Dict, List, Optional


def run_inference_and_build_submission(
    best_weights: Path,
    *,
    test_images_dir: Path,
    project_dir: Path,
    run_name: str,
    image_size: int,
    device: str,
    conf: float = 0.001,
) -> Dict[str, Path]:
    """Run Ultralytics inference and export a Kaggle-style submission CSV."""

    if not best_weights.exists():
        raise FileNotFoundError(f'Best checkpoint not found at {best_weights}')
    if not test_images_dir.exists():
        raise FileNotFoundError(f'Test images folder not found at {test_images_dir}')

    inference_model = YOLO(str(best_weights))
    predict_results = inference_model.predict(
        source=str(test_images_dir),
        project=str(project_dir),
        name=run_name,
        imgsz=image_size,
        device=device,
        conf=conf,
        save=True,
        save_txt=True,
        save_conf=True,
        exist_ok=True,
    )

    if not predict_results:
        raise RuntimeError('No predictions were returned by YOLO inference.')

    prediction_save_dir = Path(predict_results[0].save_dir)
    print(f'Inference outputs saved to: {prediction_save_dir}')

    submission_rows: List[List[str]] = []
    for result in predict_results:
        stem = Path(result.path).stem
        digits = ''.join(ch for ch in stem if ch.isdigit())
        image_id = str(int(digits)) if digits else stem
        image_h, image_w = result.orig_shape
        boxes = result.boxes
        if boxes is None or len(boxes) == 0:
            pred_string = ''
        else:
            xyxy = boxes.xyxy.cpu().numpy()
            confs = boxes.conf.cpu().numpy()
            classes = boxes.cls.cpu().numpy()
            parts: List[str] = []
            for conf, (x1, y1, x2, y2), cls in zip(confs, xyxy, classes):
                x1 = max(0.0, float(x1))
                y1 = max(0.0, float(y1))
                x2 = min(float(x2), float(image_w))
                y2 = min(float(y2), float(image_h))
                width = max(0.0, x2 - x1)
                height = max(0.0, y2 - y1)
                parts.append(f'{conf:.6f} {x1:.2f} {y1:.2f} {width:.2f} {height:.2f} {int(cls)}')
            pred_string = ' '.join(parts)
        submission_rows.append([image_id, pred_string])

    submission_rows.sort(key=lambda row: int(row[0]) if row[0].isdigit() else row[0])

    submission_path = project_dir / f'{run_name}_submission.csv'
    with submission_path.open('w', newline='') as fh:
        writer = csv.writer(fh)
        writer.writerow(['Image_ID', 'PredictionString'])
        writer.writerows(submission_rows)

    print(f'Submission file written to: {submission_path}')

    return {
        'prediction_dir': prediction_save_dir,
        'submission_path': submission_path,
    }


In [7]:
import math
import random
import shutil
from collections import Counter
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Tuple


def summarize_class_counts(records: Iterable[dict]) -> Counter[int]:
    counter: Counter[int] = Counter()
    for rec in records:
        counter.update(rec['class_hist'])
    return counter


def oversample_records(records: List[dict], target_count: Optional[int] = None, seed: int = SEED) -> Tuple[List[dict], Counter[int]]:
    """Duplicate minority samples until each class reaches ``target_count`` boxes."""

    base_counts = summarize_class_counts(records)
    if not base_counts:
        return records, Counter()
    if target_count is None:
        target_count = max(base_counts.values())

    rng = random.Random(seed)
    class_to_records = {cls: [rec for rec in records if rec['class_hist'].get(cls, 0) > 0] for cls in base_counts}
    augmented: List[dict] = list(records)
    duplicate_tracker: Dict[str, int] = {}
    current_counts = Counter(base_counts)

    progress = True
    while progress:
        progress = False
        for cls in sorted(class_to_records):
            if current_counts[cls] >= target_count:
                continue
            candidates = class_to_records[cls]
            if not candidates:
                continue
            rec = rng.choice(candidates)
            suffix = duplicate_tracker.get(rec['stem'], 0)
            duplicate_tracker[rec['stem']] = suffix + 1
            alias = f"{rec['stem']}_dup{suffix:04d}"
            augmented.append({**rec, 'alias': alias})
            for c, n in rec['class_hist'].items():
                current_counts[c] += n
            progress = True
    return augmented, current_counts


def undersample_records(records: List[dict], target_count: Optional[int] = None, seed: int = SEED) -> Tuple[List[dict], Counter[int]]:
    """Remove majority-only samples until reaching ``target_count`` boxes for the majority class."""

    base_counts = summarize_class_counts(records)
    if not base_counts:
        return records, Counter()
    majority_cls = max(base_counts, key=base_counts.get)
    if target_count is None:
        target_count = max(min(base_counts.values()), int(sum(base_counts.values()) / len(base_counts)))

    rng = random.Random(seed)
    retained = list(records)
    current_counts = Counter(base_counts)
    candidates = [rec for rec in retained if set(rec['class_hist'].keys()) == {majority_cls}]
    rng.shuffle(candidates)

    for rec in candidates:
        if current_counts[majority_cls] <= target_count:
            break
        retained.remove(rec)
        for cls, num in rec['class_hist'].items():
            current_counts[cls] -= num
    return retained, current_counts


def apply_resampling(records: List[dict], *, seed: int = SEED, oversample: bool = True, undersample: bool = True) -> Tuple[List[dict], Counter[int]]:
    """Convenience wrapper that applies under/over-sampling sequentially."""

    working_records = list(records)
    current_counts = summarize_class_counts(working_records)

    if undersample:
        working_records, current_counts = undersample_records(working_records, seed=seed)
    if oversample:
        target = max(current_counts.values()) if current_counts else None
        working_records, current_counts = oversample_records(working_records, target_count=target, seed=seed)
    return working_records, current_counts


def compute_class_weights(class_counts: Counter[int], *, method: str = 'effective_num', beta: float = 0.999) -> Dict[int, float]:
    """Derive re-weighting coefficients given class counts."""

    weights: Dict[int, float] = {}
    for cls, count in class_counts.items():
        if count <= 0:
            weights[cls] = 1.0
            continue
        if method == 'inverse':
            weights[cls] = 1.0 / float(count)
        elif method == 'sqrt_inv':
            weights[cls] = 1.0 / math.sqrt(float(count))
        else:  # effective number of samples
            weights[cls] = (1.0 - beta) / (1.0 - beta ** count)
    # Normalise so that mean weight is 1.0
    if weights:
        scale = len(weights) / sum(weights.values())
        for cls in weights:
            weights[cls] *= scale
    return weights


def materialize_dataset(records: List[dict], base_dataset_dir: Path, output_dataset_dir: Path) -> Counter[int]:
    """Create a YOLO dataset at ``output_dataset_dir`` using ``records`` for the training split."""

    if output_dataset_dir.exists():
        shutil.rmtree(output_dataset_dir)

    # Copy validation and test splits verbatim for fair comparison.
    for split in ['val', 'test']:
        src_images = base_dataset_dir / 'images' / split
        src_labels = base_dataset_dir / 'labels' / split
        dst_images = output_dataset_dir / 'images' / split
        dst_labels = output_dataset_dir / 'labels' / split
        if src_images.exists():
            shutil.copytree(src_images, dst_images, dirs_exist_ok=True)
        else:
            dst_images.mkdir(parents=True, exist_ok=True)
        if src_labels.exists():
            shutil.copytree(src_labels, dst_labels, dirs_exist_ok=True)
        else:
            dst_labels.mkdir(parents=True, exist_ok=True)

    train_images_dst = output_dataset_dir / 'images' / 'train'
    train_labels_dst = output_dataset_dir / 'labels' / 'train'
    train_images_dst.mkdir(parents=True, exist_ok=True)
    train_labels_dst.mkdir(parents=True, exist_ok=True)

    class_counter = Counter()
    for rec in records:
        alias = rec.get('alias', rec['stem'])
        shutil.copy2(rec['image_path'], train_images_dst / f'{alias}.png')
        shutil.copy2(rec['label_path'], train_labels_dst / f'{alias}.txt')
        class_counter.update(rec['class_hist'])

    return class_counter


def copy_base_dataset(base_dataset_dir: Path, output_dataset_dir: Path) -> Counter[int]:
    """Clone the baseline dataset without altering the class distribution."""

    if output_dataset_dir.exists():
        shutil.rmtree(output_dataset_dir)
    shutil.copytree(base_dataset_dir, output_dataset_dir)
    return summarize_class_counts(TRAIN_RECORDS)


def write_dataset_yaml(dataset_dir: Path, yaml_path: Path) -> None:
    data_yaml = {
        'path': dataset_dir.as_posix(),
        'train': 'images/train',
        'val': 'images/val',
        'nc': len(CLASS_NAME_MAP),
        'names': {int(k): v for k, v in CLASS_NAME_MAP.items()},
    }
    test_dir = dataset_dir / 'images' / 'test'
    if test_dir.exists() and any(test_dir.iterdir()):
        data_yaml['test'] = 'images/test'
    with yaml_path.open('w') as fh:
        yaml.safe_dump(data_yaml, fh, sort_keys=False)


In [8]:

from datetime import datetime
from dataclasses import replace
from typing import Any, Dict, Optional


def read_final_training_metrics(run_dir: Path) -> Dict[str, float]:
    """Load the last row of Ultralytics' results.csv as a metrics dictionary."""

    metrics_path = run_dir / 'results.csv'
    if not metrics_path.exists():
        return {}
    import csv

    with metrics_path.open('r', newline='') as fh:
        reader = csv.DictReader(fh)
        rows = list(reader)
    if not rows:
        return {}
    final_row = rows[-1]
    metrics: Dict[str, float] = {}
    for key, value in final_row.items():
        if key == 'epoch':
            continue
        try:
            metrics[key] = float(value)
        except (TypeError, ValueError):
            continue
    return metrics



def prepare_training_dataset(
    output_root: Path,
    *,
    resample: bool,
    seed: int,
    oversample_tail: bool = True,
    undersample_head: bool = True,
) -> Dict[str, Any]:
    """Create a YOLO-formatted dataset for the current experiment."""

    dataset_dir = output_root / 'dataset'
    staging_dir = dataset_dir.with_name(f"{dataset_dir.name}_staging")
    if staging_dir.exists():
        shutil.rmtree(staging_dir)

    if resample:
        records, _ = apply_resampling(
            TRAIN_RECORDS,
            seed=seed,
            oversample=oversample_tail,
            undersample=undersample_head,
        )
        class_counts = materialize_dataset(records, BASE_DATASET_DIR, staging_dir)
    else:
        class_counts = copy_base_dataset(BASE_DATASET_DIR, staging_dir)

    if dataset_dir.exists():
        shutil.rmtree(dataset_dir)
    staging_dir.rename(dataset_dir)

    dataset_yaml = output_root / 'longtail_dataset.yaml'
    write_dataset_yaml(dataset_dir, dataset_yaml)
    return {
        'dataset_dir': dataset_dir,
        'dataset_yaml': dataset_yaml,
        'class_counts': class_counts,
    }


def run_single_stage_pipeline(
    *,
    experiment_name: str,
    base_config: TrainConfig,
    resample: bool,
    use_class_weights: bool,
    alpha_schedule: Optional[Dict[str, float]],
    oversample_tail: bool = True,
    undersample_head: bool = True,
    output_root: Optional[Path] = None,
) -> Dict[str, Any]:
    """End-to-end train → val → infer loop for a single configuration."""

    if output_root is None:
        timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
        experiment_dir = (PROJECT_DIR / 'artifacts' / experiment_name / timestamp).resolve()
    else:
        experiment_dir = Path(output_root).resolve()
    run_root_dir = experiment_dir / 'run'
    run_root_dir.mkdir(parents=True, exist_ok=True)

    dataset_info = prepare_training_dataset(
        experiment_dir,
        resample=resample,
        seed=base_config.seed,
        oversample_tail=oversample_tail,
        undersample_head=undersample_head,
    )
    config = replace(
        base_config,
        dataset_yaml=dataset_info['dataset_yaml'],
        project_dir=run_root_dir,
        run_name='train',
        val_name='val',
    )

    class_weights = compute_class_weights(BASE_DATASET_CLASS_COUNTS) if use_class_weights else None

    train_summary = train_single_stage(
        config,
        class_weights=class_weights,
        extra_train_kwargs=None,
        initial_weights="/home/eclab314/314706007/2025-CVPDL/HW2/hw2_314706007/code_314706007/src/artifacts/yolov11_attention/20251030-131808/run/train/weights/best.pt",
        alpha_schedule=alpha_schedule,
    )
    run_validation(train_summary['model'], config)

    best_weights = train_summary['best_weights']
    inference = run_inference_and_build_submission(
        best_weights,
        test_images_dir=BASE_DATASET_DIR / 'images' / 'test',
        project_dir=run_root_dir,
        run_name='infer',
        image_size=config.image_size,
        device=config.device,
    )

    final_metrics = read_final_training_metrics(train_summary['run_dir'])

    return {
        'experiment_dir': experiment_dir,
        'dataset': dataset_info,
        'train': train_summary,
        'inference': inference,
        'metrics': final_metrics,
        'alpha_schedule': alpha_schedule,
        'class_weights_used': class_weights is not None,
    }


In [9]:

ALPHA_SCHEDULE = {
    'start': 0.05,
    'end': 1.0,
    'start_epoch': 5,
    'transition_epochs': 20,
    'power': 1.0,
}

single_stage_config = TrainConfig(
    batch_size=1,
    epochs=100,
    image_size=2560,
    workers=0,
    seed=SEED,
)

result = run_single_stage_pipeline(
    experiment_name='yolov11_attention',
    base_config=single_stage_config,
    resample=True,
    use_class_weights=True,
    alpha_schedule=ALPHA_SCHEDULE,
    oversample_tail=False,
    undersample_head=True,
    output_root=EXPERIMENT_DIR,
)

print(f"Experiment directory: {result['experiment_dir']}")
print(f"Training run directory: {result['train']['run_dir']}")
print(f"Best weights: {result['train']['best_weights']}")
print(f"Submission CSV: {result['inference']['submission_path']}")

metrics = result['metrics']
if metrics:
    print('Validation metrics (final results.csv row):')
    for key in sorted(metrics):
        value = metrics[key]
        print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")
else:
    print('Validation metrics not available.')

class_counts = result['dataset']['class_counts']
print('Training class distribution:')
for cls, count in sorted(class_counts.items()):
    print(f'  class {cls}: {count}')

print('Alpha schedule configuration:')
for key, value in ALPHA_SCHEDULE.items():
    print(f'  {key}: {value}')


Auto-selected device: cuda
Using 0 dataloader workers for training
New https://pypi.org/project/ultralytics/8.3.223 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.209 🚀 Python-3.11.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24082MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/eclab314/314706007/2025-CVPDL/HW2/hw2_314706007/code_314706007/src/artifacts/yolov10_attention/20251031-195048/longtail_dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=2560, int8=False, iou=0.7, keras=False, kobj=1.0, line

/home/eclab314/miniconda3/envs/CVPDL/lib/python3.11/site-packages/torch/autograd/graph.py:829: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:93.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/100      7.88G      2.381      1.851      1.571        125       2560: 100% ━━━━━━━━━━━━ 624/624 5.0it/s 2:05<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 95/95 7.4it/s 12.8s0.1s
                   all        190       6194      0.535      0.558       0.49      0.169
[alpha] epoch 1: 0.050

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100       7.9G      2.256        1.4      1.532         24       2560: 0% ──────────── 1/624 1.7it/s 0.4s<6:17

/home/eclab314/miniconda3/envs/CVPDL/lib/python3.11/site-packages/torch/autograd/graph.py:829: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:93.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/100       7.9G      2.315      1.593      1.556         44       2560: 100% ━━━━━━━━━━━━ 624/624 5.2it/s 2:01<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 95/95 9.1it/s 10.5s0.1s
                   all        190       6194      0.564      0.567      0.526      0.182
[alpha] epoch 2: 0.050

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      7.93G      1.723       1.17      1.215         43       2560: 0% ──────────── 1/624 1.6it/s 0.4s<6:30

...

/home/eclab314/miniconda3/envs/CVPDL/lib/python3.11/site-packages/torch/autograd/graph.py:829: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:93.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     33/100      11.2G      2.073      1.244      1.434         76       2560: 100% ━━━━━━━━━━━━ 624/624 5.3it/s 1:58<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 95/95 8.9it/s 10.6s0.1s
                   all        190       6194      0.604      0.606      0.603      0.219
[alpha] epoch 33: 1.000

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     34/100      11.3G      2.216      1.471      1.748         26       2560: 0% ──────────── 1/624 1.7it/s 0.4s<6:11

/home/eclab314/miniconda3/envs/CVPDL/lib/python3.11/site-packages/torch/autograd/graph.py:829: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:93.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     34/100      11.3G      2.061      1.205      1.435         37       2560: 14% ━╸────────── 86/624 5.3it/s 16.6s<1:41


KeyboardInterrupt: 